In [23]:
import pandas as pd
import numpy as np
import requests
import re
import asyncio
import aiohttp

from tqdm import tqdm

tqdm.pandas()

In [3]:
def get_type_by_request(prefix):
    url = f"https://hdl.handle.net/api/handles/{prefix}"
    response = requests.get(url)

    try:
        data = response.json()

        for val in data['values']:
            if val['type'] == 'HS_SERV':
                return val['data']['value']
        return np.nan

    except:
        return np.nan

In [ ]:
async def fetch(session, url):
    async with session.get(url) as resp:
        return await resp.json()

In [ ]:
async def main():
    async with aiohttp.ClientSession() as session:
        tasks = [fetch(session, url) for url in urls]
        results = await asyncio.gather(*tasks)
    return results

In [2]:
prefixes = [f'10.{i}' for i in range(0, 100000)]
all_prefixes = pd.DataFrame({'prefix': prefixes})

In [3]:
urls = [f"https://hdl.handle.net/api/handles/10.{i}" for i in range(0, 100000)]
data = await main()

In [5]:
result = []
for json in tqdm(data):
    try:
        type_cit = ''
        for val in json['values']:
            if val['type'] == 'HS_SERV':
                type_cit += val['data']['value']
                
        if len(type_cit) == 0:
            type_cit = np.nan
        result.append(type_cit)
    except:
        result.append(np.nan)

100%|██████████| 100000/100000 [00:00<00:00, 391013.13it/s]


In [6]:
descr = []
for json in tqdm(data):
    try:
        desc = ''
        for val in json['values']:
            if val['type'] == 'DESC':
                desc += val['data']['value']

        if len(desc) == 0:
            desc = np.nan
        descr.append(desc)
    except:
        descr.append(np.nan)

100%|██████████| 100000/100000 [00:00<00:00, 377494.87it/s]


In [15]:
all_prefixes['type'] = result
all_prefixes['description'] = descr
all_prefixes.dropna(subset = ['type'], inplace = True)

In [45]:
unknown = all_prefixes[all_prefixes['description'].progress_apply(lambda info: re.search(re.compile(r'data\s*cite', re.IGNORECASE), info) is not None if type(info) is str else False)]

100%|██████████| 78095/78095 [00:00<00:00, 407564.35it/s]


In [56]:
unknown = unknown[unknown['type'] != '10.SERV/DATACITE']
all_prefixes.loc[unknown.index, ['type']] = '10.SERV/MIXED'
all_prefixes.drop(columns = ['description']).to_csv('prefixes.csv', index = False)